# Week 5 へようこそ - エージェントフレームワーク

## Day 3: Microsoft Agent Framework

同じ週、同じ5つのステップ、新しいフレームワークです。今週の考え方全体は、ひとつのエージェントフレームワークを理解すれば他もだいたい理解できるということなので、毎日同じ5つのステップで同じエージェントを作り、その作法が響き合う様子を観察します。

1. **エージェントを作る** - モデルとシステムプロンプトを与える。
2. **実行する** - メッセージを送り、返信を受け取る。
3. **ツールを追加する** - エージェントが呼び出せる、普通の型付き関数。
4. **MCP を追加する** - 誰か他の人が書いたツールサーバーに接続する。毎回同じ方法でつなげる。
5. **ゴールを与えてループさせる** - 目標を渡し、仕事が終わるまで一歩ずつ自分で進めさせる。

ステップ1と2は、まだ単なる LLM 呼び出しです。ツールと MCP は、エージェントにできることを与えます。ステップ5でようやくエージェントらしくなります。フレームワーク自身がループを回し、ツールを選び、結果を読み、また選び直す、というのをゴールに到達するまで続けるのです。

実習プロジェクトは Day 1 と同じ SQLite の todo ボードです。ワーカーがボードから1つのゴールを取り出し、自分でステップを計画し、自分のエージェントループでその作業をこなし、各ステップにチェックを入れていきます。ボードのコード(`board.py`)は一字一句まったく同じファイルで、変わるのはそれを取り巻くフレームワークだけです。

今日は **Microsoft Agent Framework** です。Microsoft の公式エージェント SDK です。知っておく価値のある背景があります。Microsoft は長年、マルチエージェント研究向けの AutoGen と、本番運用の配管向けの Semantic Kernel という2つの別々のエージェント関連プロジェクトを走らせてきましたが、このフレームワークはその2つが1つに収束したものです。Python でも C# でもほとんど同じ見た目の、たった1つの `Agent` が手に入り、その裏では、長時間にわたる耐久性のあるプロセスのための、グラフベースのワークフローエンジンが動いています。ここでは素のエージェントを使い、最後にワークフローエンジンを見ていきます。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Microsoft Agent Framework のドキュメント</h2>
            <span style="color:#00bfff;">ドキュメントは <a href="https://learn.microsoft.com/en-us/agent-framework/">learn.microsoft.com/agent-framework</a> にあります。このフレームワークは2026年4月に 1.0 に達し、その過程で API の名前を大きく変更したので、ここではバージョンを固定(1.8.1)し、古いブログ記事は注意して扱ってください。知っておくべき2つの形として、OpenAI クライアントはモデルを <code>model=</code> として受け取り、<code>Agent</code> は <code>client</code> と <code>instructions</code> から作られます。</span>
        </td>
    </tr>
</table>

## セットアップ

今日必要なものは2つですが、どちらも以前の週からすでに用意されています。

- **Node**。`npx` のために必要です(filesystem MCP サーバーはこれを使って動きます)。`node --version` で確認してください。
- リポジトリのルートにある `.env` の中の **`GOOGLE_API_KEY`**。今日のフレームワークは Gemini の `gemini-flash-latest` を、GeminiのOpenAI互換エンドポイント経由で使います。

Microsoft Agent Framework はリポジトリの環境に含まれているので、リポジトリのルートで通常の `uv sync` を実行すればすべてインストールされます。このノートブックを Cursor で開き、毎週使っているリポジトリ既定の **Python 3.12.12** カーネルを選んで、上から順にセルを実行してください。

最初の実行を速くするために、今のうちに一度 filesystem MCP サーバーをウォームアップしておき、動作中と表示されたらすぐに Ctrl-C で止めてください。

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [ ]:
# このフレームワークはインポート時に「experimental」の通知をいくつか表示するので、それを抑える。
import warnings
warnings.filterwarnings("ignore", message=r".*experimental.*")

In [ ]:

import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agent_framework import Agent, MCPStdioTool
from agent_framework.openai import OpenAIChatClient

load_dotenv(override=True)


## ステップ1: エージェントを作る

Microsoft Agent Framework では、エージェントは `Agent` です。モデル用の `client` と、システムプロンプトである `instructions` を持ちます。ここではクライアントを、GeminiのOpenAI互換エンドポイントを指す `OpenAIChatClient` として一度だけ組み立て、環境変数から `GOOGLE_API_KEY` を読み込み、それをどこでも再利用します。このフレームワークは非同期を前提としているので、エージェントは常に `await` します。

In [ ]:
MODEL = "gemini-flash-latest"

# メインモデルをOpenAIからGeminiに切り替え(GeminiのOpenAI互換エンドポイント経由)
client = OpenAIChatClient(
    model=MODEL,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ["GOOGLE_API_KEY"],
)

agent = Agent(
    client=client,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)

## ステップ2: 実行する

メッセージを送り、返信を待ち、結果の `.text` を表示します。まだツールがないので、ループするものは何もなく、エージェントはただ答えるだけです。これはまだ単なる LLM 呼び出しです。

In [ ]:
result = await agent.run("Say hello in Spanish.")
print(result.text)

## 今週のプロジェクト: SQLite の todo ボード

ワーカーは、Day 1 と同じ小さな SQLite ボード、同じ `board.py` ファイルを介して連携します。1つのファイル、1つのテーブルで、サーバーを立てる必要もありません。ワーカーには1つの**ゴール**が与えられ、それを達成するために自分自身の**ステップ**の todo をそのゴールの下に書き出し、進めるごとにチェックを入れていき、最後にゴールを完了にします。内部的にはボードは単なる辞書のリストです(ゴールの `parent_id` は None で、ステップは自分のゴールを指します)。

In [ ]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

`show_board()` は、Week 1 で使ったのと同じ rich スタイルで、その同じデータを綺麗に表示します。各ゴールの下にステップがインデントされて並び、完了したタスクは緑色の打ち消し線、進行中のタスクは黄色で表示されます。まだステップはありません。エージェントが計画を立てるときに自分でステップを書き出します。

In [ ]:
board.show_board()

## ステップ3: ツールを追加する

Microsoft Agent Framework でのツールは、普通の型付き Python 関数です。フレームワークが型ヒントと docstring を読み取り、JSON スキーマを自動で組み立ててくれるので、他に宣言することは何もありません。`@tool` デコレーターは、何かの名前を変えたり制約を加えたりしたい場合にのみ使います。関数はエージェントの `tools=[...]` リストに渡します。

ここでは3つの小さなボードツールを書きます。ボードを読む `show_todos`、ゴールをステップに分解する `plan_steps`、todo を完了にする `complete_task` です。まずは簡単なエージェントに2つだけ与えて、ボードに何があるか尋ねてみましょう。答える前に自分から `show_todos` を呼び出すことを、自分の目で確かめてください。この「決める、呼ぶ、読む、答える」というサイクルこそ、エージェントループが回り始めた瞬間です。3つのツールすべてはステップ5で一緒になります。

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
board_agent = Agent(
    client=client,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [ ]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.text)

## ステップ4: MCP を追加する

MCP は、単に「自分が書いていないツール」を、小さなプロトコル越しに接続したものです。今週すべてのフレームワークで使う同じ Node サーバーである filesystem リファレンスサーバーを、単一の `workspace` フォルダに限定してエージェントに与えます。これにより、エージェントはそのフォルダ内のファイルしか触れなくなります。Microsoft Agent Framework では、MCP サーバーは `MCPStdioTool` です。`async with` で開き、同じ `tools=[...]` リストでエージェントに渡します。

このフレームワークの `MCPStdioTool` はサーバーの stderr を公開していないので、これをサブクラス化して、内部の stdio クライアントに2つの設定を行います。`errlog=subprocess.DEVNULL` はサーバーの起動時バナーを静かにし、Windows 上の Jupyter カーネルからサーバーを実行できるようにします。Windows のカーネルの stderr には実際のファイルディスクリプタがないためです。`cwd` はサーバーを workspace 内で起動するので、エージェントのファイル名はそこで解決されます。Mac と Linux では errlog は単純に出力を綺麗に保つだけです。

In [ ]:
workspace = Path("workspace").resolve()   # エージェントが触れてよい唯一のフォルダ


class FilesystemMCP(MCPStdioTool):
    """filesystem サーバーの stderr を DEVNULL に送り、作業ディレクトリを workspace に
    設定したもの。これによりファイル名がそこで解決されるようになり、Windows 上の
    Jupyter カーネルからも問題なく実行できる。"""

    def get_mcp_client(self):
        from mcp.client.stdio import StdioServerParameters, stdio_client

        params = StdioServerParameters(command=self.command, args=self.args, env=self.env, cwd=str(workspace))
        return stdio_client(server=params, errlog=subprocess.DEVNULL)


filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
)

In [ ]:
file_agent = Agent(
    client=client,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem],
)

In [ ]:
async with filesystem:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
print(result.text)

## ステップ5: ゴールを与えてループさせる

さあ、いよいよ本番です。1つのエージェントに3つのボードツールすべてと filesystem サーバーを与え、ゴールを渡して、実行させましょう。エージェントは自分でボード上にステップを計画し、ファイルツールでそれを片付け、それぞれにチェックを入れ、作業が終わったらゴールを完了にします。これこそ、自律的に動くエージェントループです。読む、計画する、行動する、チェックする、繰り返す。ボードにステップが埋まり、それが打ち消し線で消されていく様子を観察してください。

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

async with filesystem:
    await worker.run("Please work the pending goal on the board.")
board.show_board()

## 同じワーカーをターミナルから実行する

ステップ5でたった今観察した内容はすべて、このノートブックの隣にある小さなスクリプト `maf_worker.py` としてもパッケージ化されています。これは同じゴールを登録し、同じ3つのボードツールと filesystem MCP サーバーを使って同じエージェントを組み立て、カーネルではなくコマンドラインから同じループを実行します。このフォルダでターミナルを開いて実行してください。

```bash
uv run maf_worker.py
```

エージェントがステップを計画し、ゴールに取り組んでそれぞれにチェックを入れていく様子、そして完成したボードと、書き出されたスペイン語が表示されます。これは Day 5 でどのワーカーも取る形と同じです。Day 5 では、Google ADK のオーケストレーターが、フレームワークごとにこうしたワーカーを1つずつ、共有された1つのボードに対する並列サブプロセスとして起動します。

## いちばん興味深い点: 耐久性のあるワークフローと、2つの言語で1つの API

素の `Agent` は入り口にすぎません。企業がこのフレームワークを本気で検討する理由は、その下にある層、グラフベースの**ワークフローエンジン**にあります。エージェントと普通の関数を、型付きのノードとエッジとして組み合わせて配線すると、フレームワークはそれをチェックポイント、ストリーミング、人間参加型の承認を備えた、耐久性があり再起動可能なプロセスとして実行します。これは AutoGen と Semantic Kernel の統合が生んだ具体的な成果であり、ジョブが再起動を乗り越えなければならないほど長く動く場合に頼るべきものです。ここでは実行しませんが、おおよその形は次のようになります。

```python
from agent_framework import WorkflowBuilder

workflow = (
    WorkflowBuilder()
    .add_edge(researcher, writer)     # 各ノードはエージェントか普通の関数
    .add_edge(writer, reviewer)
    .build()
)
```

同じ `Agent` と `OpenAIChatClient` の形は C# にも存在するので、チームは設計をほぼ1行単位で Python と .NET の間を移動させることができます。そして、今週の他の部分と同じワンライン精神で、`OpenAIChatClient` は `base_url` を受け取れるので、任意の OpenAI 互換エンドポイントで動かすには、そこを指定するだけで他は何も変わりません。

```python
client = OpenAIChatClient(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY, model="openai/gpt-5.4-mini")
```

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">エクササイズ</h2>
            <span style="color:#ff7800;">ボードに別のゴールを、たとえば「マドリードについての短い俳句を書いて madrid.txt に保存する」を登録し、ワーカーを再度実行してみましょう。ワーカーは適切なステップを計画し、正しいファイルツールを選べるでしょうか。次に、<code>base_url</code> を渡して <code>OpenAIChatClient</code> を、自分がアクセスできる別の OpenAI 互換エンドポイントに向け、ワーカーを再実行し、同じエージェントが別のモデルで動く様子を観察してみましょう。</span>
        </td>
    </tr>
</table>